# Greeks Hedging

Options hedging involves neutralizing sensitivities (Greeks). 

### Core Theory
To eliminate risk: $\text{Instruments} \times \text{Sizes} = -\text{Portfolio Greeks}$

In [1]:
import numpy as np
portfolio = np.array([150.0, 25.0, 40.0, -60.0, 15.0])
instruments = np.array([
    [ 0.55,  0.40, -0.35,  0.20,  1.0],
    [ 0.06,  0.04,  0.05,  0.02,  0.0],
    [ 0.08,  0.05,  0.07,  0.03,  0.0],
    [-0.12, -0.08, -0.10, -0.05,  0.0],
    [ 0.04,  0.03, -0.02,  0.01,  0.0]
])
weights = np.linalg.solve(instruments, -portfolio)
names = ["Option 1", "Option 2", "Option 3", "Option 4", "Underlying Stock"]
for name, w in zip(names, weights):
    action = "LONG" if w >= 0 else "SHORT"
    print(f"{name}: {action} Size: {abs(w):.2f}")

Option 1: SHORT Size: 900.00
Option 2: LONG Size: 1100.00
Option 3: LONG Size: 100.00
Option 4: SHORT Size: 1000.00
Underlying Stock: LONG Size: 140.00


# Overdetermined System: 5 Greeks, 4 Options

### Theory
More risk factors than hedging options. Minimize unhedged risk using Ordinary Least Squares (OLS): $\min_{w} \| Xw - y \|^2$

In [2]:
options = np.array([
    [ 0.55,  0.40, -0.35,  0.20],
    [ 0.06,  0.04,  0.05,  0.02],
    [ 0.08,  0.05,  0.07,  0.03],
    [-0.12, -0.08, -0.10, -0.05],
    [ 0.04,  0.03, -0.02,  0.01]
])
w_options, residuals, rank, s = np.linalg.lstsq(options, -portfolio, rcond=None)
print("Least-Squares Weights:", np.round(w_options, 2))

Least-Squares Weights: [-184.38 -217.26 -154.25  -79.45]


# Condition Number

### Theory
Condition Number (Kappa) measures structural equation stability. Values $<1000$ are stable, while values $>1000$ are ill-conditioned.

In [3]:
cond_num = np.linalg.cond(instruments)
print(f"Condition Number: {cond_num:.2f}")

Condition Number: 642.00


# Multicollinearity and Regularization

### Theory
Identical choices cause explosive trade configurations. Fix via Ridge Regularization ($L_2$ Penalty): $\min_{w} \| Xw - y \|^2 + \alpha \| w \|^2$

In [4]:
from sklearn.linear_model import Ridge
options_matrix = np.array([
    [0.06, 0.059, 0.05, 0.02],
    [0.08, 0.079, 0.07, 0.03],
    [-0.12, -0.119, -0.10, -0.05],
    [0.04, 0.039, -0.02, 0.01]
])
portfolio_targets = np.array([-25.0, -40.0, 60.0, -15.0])
w_options_pinv = np.linalg.pinv(options_matrix) @ portfolio_targets
ridge_model = Ridge(alpha=0.1, fit_intercept=False)
ridge_model.fit(options_matrix, portfolio_targets)
print("Pinv Weights:", np.round(w_options_pinv, 2))
print("Ridge Weights:", np.round(ridge_model.coef_, 2))

Pinv Weights: [ -60.76 -131.87   -5.87 -741.51]
Ridge Weights: [-73.29 -72.47 -56.48 -28.43]


# Constrained Optimization

### Theory
Balances Greek errors with hard position boundary rules and absolute trading volume costs ($L_1$ Penalty): $\min_{w} \sum (\text{Greek Error})^2 + \lambda \sum |w|$

In [5]:
from scipy.optimize import minimize
def objective_function(weights, matrix, targets, tx_cost_penalty=0.5):
    resulting_greeks = matrix @ weights
    mismatch_error = np.sum((resulting_greeks - targets) ** 2)
    tx_costs = tx_cost_penalty * np.sum(np.abs(weights))
    return mismatch_error + tx_costs

bounds = [(-150, 150) for _ in range(4)]
result = minimize(objective_function, np.zeros(4), args=(options_matrix, portfolio_targets, 0.5), bounds=bounds, method='SLSQP')
print("Smart Bounded Weights:", np.round(result.x, 1))

Smart Bounded Weights: [-150.  -150.  -150.  -110.6]
